## Задание
*Для изучения CSS селекторов https://flukeout.github.io/*

- Найдите сайт с открытыми таблицами (например, статистика IMDB, погодные данные, вакансии) и с помощью requests + BeautifulSoup извлеките несколько колонок и загрузите в DataFrame.

- Установите заголовок запроса (headers={'User-Agent': 'Mozilla/5.0 ...'}) и добавьте time.sleep(1) между запросами при парсинге нескольких страниц.

- Используйте soup.select() с CSS-селектором, чтобы выбрать интересующие элементы и извлечь текст или атрибуты.

- После получения данных с веба отметьте, как часто встречаются пропущенные значения или некорректные форматы (например, строка вместо числа)  и примените методы Pandas по очистке.

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 1. Загружаем страницу с headers
url = "https://realpython.github.io/fake-jobs/"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
html = response.text

# 2. Делаем soup
soup = BeautifulSoup(html, "html.parser")

# 3. Используем soup.select() с CSS-селектором
# Каждая вакансия лежит в div с классом "card-content"
job_cards = soup.select("div.card-content")

data = []

for card in job_cards:
    # вытаскиваем заголовок, компанию, местоположение и дату
    title_el = card.select_one("h2.title")
    company_el = card.select_one("h3.company")
    location_el = card.select_one("p.location")
    date_el = card.select_one("time")

    title = title_el.get_text(strip=True) if title_el else None
    company = company_el.get_text(strip=True) if company_el else None
    location = location_el.get_text(strip=True) if location_el else None
    date_posted = date_el.get_text(strip=True) if date_el else None

    data.append({
        "Должность": title,
        "Компания": company,
        "Локация": location,
        "Дата публикации": date_posted
    })

    # если бы мы обходили несколько страниц, делали бы паузу
    time.sleep(0.1)

# 4. Загружаем данные в DataFrame
df = pd.DataFrame(data)
print("Первые строки исходного DataFrame:")
print(df.head(), "\n")

# 5. Проверяем пропуски
print("Пропуски по столбцам:")
print(df.isna().sum(), "\n")

# 6. Пример проверки некорректных форматов:
# допустим, хотим убедиться, что 'Дата публикации' строка, а не число
bad_dates = df[~df["Дата публикации"].apply(lambda x: isinstance(x, str))]
print("Строки с некорректным форматом даты (если есть):")
print(bad_dates, "\n")

# 7. Очистка: удалим строки, где нет должности или компании
df_clean = df.dropna(subset=["Должность", "Компания"])
print("Чистый DataFrame после очистки:")
print(df_clean.head())

Первые строки исходного DataFrame:
                 Должность                    Компания               Локация  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

  Дата публикации  
0      2021-04-08  
1      2021-04-08  
2      2021-04-08  
3      2021-04-08  
4      2021-04-08   

Пропуски по столбцам:
Должность          0
Компания           0
Локация            0
Дата публикации    0
dtype: int64 

Строки с некорректным форматом даты (если есть):
Empty DataFrame
Columns: [Должность, Компания, Локация, Дата публикации]
Index: [] 

Чистый DataFrame после очистки:
                 Должность                    Компания               Локаци